### Deep Agents overview



Build agents that can plan, use subagents, and leverage file systems for complex tasks

deepagents is a standalone library for building agents that can tackle complex, multi-step tasks. Built on LangGraph and inspired by applications like Claude Code, Deep Research, and Manus, deep agents come with planning capabilities, file systems for context management, and the ability to spawn subagents.


### When to use deep agents
Use deep agents when you need agents that can:

- Handle complex, multi-step tasks that require planning and decomposition
- Manage large amounts of context through file system tools
- Delegate work to specialized subagents for context isolation
- Persist memory across conversations and threads

In [17]:
### Basic deep agent

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")

In [18]:
from typing import Literal

In [19]:
### Tools- Internet search
from tavily import TavilyClient # type: ignore
from typing import Literal

tavily_client=TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

def web_search(query:str,max_results:int=5,
topic: Literal["general","sports","news","finance"]="general",
include_raw_content:bool=False):
    """Run a web search"""
    return tavily_client.search(query,
    max_results=max_results,include_raw_content=include_raw_content,topic=topic)



In [4]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023788AFF4A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023788B0A450>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
## Basic Agent
from langchain.agents import create_agent


simple_agent=create_agent(
    model=model,
    tools=[web_search]
)
simple_agent

NameError: name 'web_search' is not defined

In [2]:
### Create a deep agent
## Prompt

## agent

from deepagents import create_deep_agent

deepagent=create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt="Act as a researcher"
)
deepagent

NameError: name 'model' is not defined

In [37]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [36]:
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

for m in client.models.list().data:
    print(m.id)

groq/compound
canopylabs/orpheus-v1-english
llama-3.1-8b-instant
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-20b
whisper-large-v3-turbo
llama-3.3-70b-versatile
openai/gpt-oss-safeguard-20b
groq/compound-mini
meta-llama/llama-prompt-guard-2-86m
allam-2-7b
openai/gpt-oss-120b
qwen/qwen3.6-27b
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3


In [40]:
result = deepagent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is deepagent?"
        }
    ]
})

print(result["messages"][-1].content)

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=write_todos>[{"content": "Define deepagent", "status": "pending"}, {"content": "Explain deepagent capabilities", "status": "pending"}, {"content": "Provide examples of deepagent usage", "status": "pending"}]</function>'}}

In [41]:
result["messages"][-1].content

NameError: name 'result' is not defined

In [42]:
result['files']

NameError: name 'result' is not defined

### What happened?

Your deep agent automatically:
- Planned its approach: Used the built-in write_todos tool to break down the research task
- Conducted research: Called the internet_search tool to gather information
- Managed context: Used file system tools (write_file, read_file) to offload large search results
- Spawned subagents (if needed): Delegated complex subtasks to specialized subagents
- Synthesized a report: Compiled findings into a coherent response

### Customizing Deep Agents

In [1]:
## #Model
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent

model = init_chat_model(model="gpt-5")
agent = create_deep_agent(model=model)
agent

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is deepagent?"}]})
result

### System Prompt
Deep agents come with a built-in system prompt inspired by Claude Code’s system prompt. The default system prompt contains detailed instructions for using the built-in planning tool, file system tools, and subagents.
Each deep agent tailored to a use case should include a custom system prompt specific to that use case.

In [49]:
from deepagents import create_deep_agent

research_instructions = """\
You are an expert researcher. Your job is to conduct \
thorough research, and then write a polished report. \
"""

agent = create_deep_agent(
    model=model,
    system_prompt=research_instructions,
)
result = agent.invoke({"messages": [{"role": "user", "content": "What is deepagent?"}]})
result

KeyboardInterrupt: 

In [ ]:
agent

In [48]:
### Tools
import os
from typing import Literal
from tavily import TavilyClient
from deepagents import create_deep_agent

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

agent = create_deep_agent(
    model=model,
    tools=[internet_search]
)
result = agent.invoke({"messages": [{"role": "user", "content": "What is deepagent in Agentic AI?"}]})
result

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=write_todos>[{"content": "Define deepagent", "status": "pending"}, {"content": "Explain role of deepagent in Agentic AI", "status": "pending"}, {"content": "Provide examples of deepagent tasks", "status": "pending"}]</function>'}}

In [47]:
import sys
print(sys.executable)

d:\Deep-agents-With-Langchain\.venv\Scripts\python.exe


In [46]:
!{sys.executable} -m pip show tavily-python

'{sys.executable}' is not recognized as an internal or external command,
operable program or batch file.


In [45]:
!{sys.executable} -m pip list | findstr tavily

'{sys.executable}' is not recognized as an internal or external command,
operable program or batch file.


In [44]:
import pkgutil
print(any(m.name=="tavily" for m in pkgutil.iter_modules()))

True
